# Municipal Candidate Data

In order to analyze the financial data, I want additional metrics - the number of candidates in each municipality and the amount of votes. The easiest way I can see to produce this is to use the Ministry of Justice voting data by candidates and reducing them to municipalities only.

In [3]:
import pandas as pd
pd.set_option('display.max_columns', None)

kuntavaalit_ehdokas_sarakkeet = list(pd.read_csv("files/Results_title_rows_EN_ehdokas.csv"))

years_data = {
    2012: {
        'results': "files/election_results_by_candidate/kv-2012_teat_maa.csv"
    },
    2017: {
        'results': "files/election_results_by_candidate/kv-2017_teat_maa.csv"
    },
    2021: {
        'results': "files/election_results_by_candidate/kv-2021_teat_maa.csv"
    },
    2025: {
        'results': "files/election_results_by_candidate/kv-2025_teat_maa.csv"
    }
}

def process_voting_data(years_data):
    """Function for preparing voting data to be municipal results only."""

    all_years_data = []
    for year, paths in years_data.items():
        print(f"Processing {year}...")

        kuntavaalit_ehdokas = pd.read_csv(paths['results'], sep=";", on_bad_lines='warn', encoding='latin-1', names=kuntavaalit_ehdokas_sarakkeet, index_col=False)

        kuntavaalit_ehdokas['Year'] = str(year) # Year column for classification later

        kuntavaalit_ehdokas["LR_cand-id"] = (
            kuntavaalit_ehdokas["Candidate number"].astype(str).str.strip() + "-" +
            kuntavaalit_ehdokas["Name of a municipality/electoral district/voting area in Finnish"].str.strip() + "-" + kuntavaalit_ehdokas["Area type"] + "-" + kuntavaalit_ehdokas['Year']
        )
        all_years_data.append(kuntavaalit_ehdokas)

    results = pd.concat(all_years_data, ignore_index=True)
    results = results.drop(results[results['Area type']!="K"].index) # K stands for the total municipal election
    return results

kuntavaalit_results = process_voting_data(years_data)
display(kuntavaalit_results)

# # Combine all years into one dataframe
#
# print(f"In total, there were {len(all_years_data[0])} candidates recorded from 2012; {len(all_years_data[1])} from 2017; {len(all_years_data[2])} from 2021; and {len(all_years_data[3])} from 2025.")

Processing 2012...


FileNotFoundError: [Errno 2] No such file or directory: 'files/election_results_by_candidate/kv-2012_teat_maa.csv'

In [1]:
print(kuntavaalit_results["Total number of votes"].describe())

NameError: name 'kuntavaalit_results' is not defined

In [23]:
kuntavaalit_municipal_candidates = kuntavaalit_results.groupby(['Name of a municipality/electoral district/voting area in Finnish', 'Year']).agg(num_candidates=('Year', 'size'), total_votes=('Total number of votes', 'sum')).reset_index()
kuntavaalit_municipal_candidates["Name of a municipality/electoral district/voting area in Finnish"] = kuntavaalit_municipal_candidates["Name of a municipality/electoral district/voting area in Finnish"].str.strip()

kuntavaalit_results.to_csv("files/outputs/municipal_election_results_by_candidate.csv")
kuntavaalit_municipal_candidates.to_csv("files/outputs/municipal_macro_per_year.csv")

display(kuntavaalit_municipal_candidates)

,Name of a municipality/electoral district/voting area in Finnish,Year,num_candidates,total_votes
0,Akaa,2012,157,7590
1,Akaa,2017,134,7532
2,Akaa,2021,164,6867
3,Akaa,2025,132,6950
4,Alajärvi,2012,112,5384
...,...,...,...,...
1179,Ähtäri,2025,75,2341
1180,Äänekoski,2012,144,9201
1181,Äänekoski,2017,126,8647
1182,Äänekoski,2021,124,7650
